# AI-Based Learning Style Report - Web Compatible Model

Notebook ini dibuat agar model `.pkl` cocok dengan web FastAPI + HTML/CSS/JS yang menerima 3 fitur:

- `visual`
- `auditory`
- `kinesthetic`

Output akhir notebook ini adalah:

`models/model_gaya_belajar.pkl`

File tersebut bisa langsung dipakai oleh `main.py`.

## 1. Import Library

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
LABEL_NAMES = {
    "V": "Visual",
    "A": "Auditory",
    "K": "Kinesthetic",
}

print("Library siap digunakan")

## 2. Load Dataset

Jika notebook berada di folder `AI-Based-Learning-Style-Report`, dataset lama biasanya ada satu folder di atasnya sebagai `../SL_csv.csv`.

In [ ]:
DATA_PATH = Path("../SL_csv.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("SL_csv.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Dataset SL_csv.csv tidak ditemukan. Letakkan di folder project atau satu folder di atas project.")

df_raw = pd.read_csv(DATA_PATH)
print("Dataset:", DATA_PATH.resolve())
print("Shape awal:", df_raw.shape)
df_raw.head()

## 3. Rename Kolom dan Mapping Pertanyaan

Dataset asli memiliki 15 pertanyaan. Mapping yang dipakai:

- Q1-Q5: Visual / Reading
- Q6-Q10: Auditory
- Q11-Q15: Kinesthetic

In [ ]:
question_columns_original = list(df_raw.columns[2:-1])
if len(question_columns_original) != 15:
    raise ValueError(f"Jumlah kolom pertanyaan harus 15, ditemukan {len(question_columns_original)}")

rename_map = {column: f"Q{i + 1}" for i, column in enumerate(question_columns_original)}
df = df_raw.rename(columns=rename_map).copy()

q_cols = [f"Q{i}" for i in range(1, 16)]
visual_cols = q_cols[0:5]
auditory_cols = q_cols[5:10]
kinesthetic_cols = q_cols[10:15]

print("Visual columns     :", visual_cols)
print("Auditory columns   :", auditory_cols)
print("Kinesthetic columns:", kinesthetic_cols)

## 4. Data Cleaning

In [ ]:
required_columns = q_cols + ["Learner"]
df = df[required_columns].copy()

before = len(df)
df = df.drop_duplicates()
print(f"Drop duplikat: {before} -> {len(df)}")

df = df[df["Learner"].isin(["V", "A", "K"])].copy()

for col in q_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=q_cols + ["Learner"])
print(f"Drop NaN: {before} -> {len(df)}")

in_range = ~((df[q_cols] < 1) | (df[q_cols] > 5)).any(axis=1)
before = len(df)
df = df[in_range].reset_index(drop=True)
print(f"Filter skor 1-5: {before} -> {len(df)}")

print("Distribusi label:")
print(df["Learner"].value_counts())

## 5. Feature Engineering untuk Web

Web mengirim rata-rata skor 1-5 untuk setiap dimensi. Karena itu model juga dilatih menggunakan rata-rata:

- `visual`
- `auditory`
- `kinesthetic`

In [ ]:
df["visual"] = df[visual_cols].mean(axis=1).round(3)
df["auditory"] = df[auditory_cols].mean(axis=1).round(3)
df["kinesthetic"] = df[kinesthetic_cols].mean(axis=1).round(3)

FEATURE_COLS = ["visual", "auditory", "kinesthetic"]
TARGET_COL = "Learner"

df[FEATURE_COLS + [TARGET_COL]].head()

## 6. Train/Test Split

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

classes = np.array(["A", "K", "V"])
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Class weights:", class_weight_dict)

## 7. Train dan Bandingkan Model

Notebook ini membandingkan Logistic Regression dan Random Forest. Model terbaik akan diekspor.

In [ ]:
candidate_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight=class_weight_dict,
            max_iter=1000,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in candidate_models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)

    results.append({
        "name": name,
        "model": model,
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "test_accuracy": test_accuracy,
        "y_pred": y_pred,
    })

    print(f"{name}")
    print(f"  CV accuracy  : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
    print(f"  Test accuracy: {test_accuracy:.4f}")
    print()

best_result = sorted(results, key=lambda item: (item["test_accuracy"], item["cv_mean"]), reverse=True)[0]
best_model = best_result["model"]

print("Model terbaik:", best_result["name"])
print("Test accuracy:", round(best_result["test_accuracy"], 4))

## 8. Evaluasi Model Terbaik

In [ ]:
print(classification_report(y_test, best_result["y_pred"], target_names=[LABEL_NAMES[c] for c in ["A", "K", "V"]]))
print("Confusion matrix labels: A, K, V")
print(confusion_matrix(y_test, best_result["y_pred"], labels=["A", "K", "V"]))

## 9. Simulasi Payload dari Web

Contoh ini sama dengan format yang dikirim `script.js` ke endpoint `/predict`.

In [ ]:
web_payload = {
    "visual": 4.33,
    "auditory": 2.67,
    "kinesthetic": 3.00,
}

sample = pd.DataFrame([web_payload], columns=FEATURE_COLS)
prediction = best_model.predict(sample)[0]

print("Payload web:", web_payload)
print("Prediksi raw:", prediction)
print("Prediksi label:", LABEL_NAMES[prediction])

if hasattr(best_model, "predict_proba"):
    probabilities = best_model.predict_proba(sample)[0]
    print("Probabilitas:")
    for label, probability in zip(best_model.classes_, probabilities):
        print(f"  {LABEL_NAMES[label]}: {probability:.2%}")

## 10. Export Model ke Folder Web

Setelah cell ini dijalankan, file berikut akan tersedia:

`models/model_gaya_belajar.pkl`

FastAPI `main.py` akan otomatis memakai file ini.

In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "model_gaya_belajar.pkl"
METADATA_PATH = MODEL_DIR / "model_metadata.json"

joblib.dump(best_model, MODEL_PATH)

metadata = {
    "project": "AI-Based Learning Style Report: Sistem Kuesioner VAK",
    "team": ["Abraham Gomes Samosir", "Tristan Bonardo Silalahi", "Newten Putra Santoso"],
    "university": "Universitas Padjadjaran (UNPAD)",
    "model_name": best_result["name"],
    "features": FEATURE_COLS,
    "target_labels": LABEL_NAMES,
    "test_accuracy": float(best_result["test_accuracy"]),
    "input_format": "FastAPI /predict expects JSON: {'visual': float, 'auditory': float, 'kinesthetic': float}",
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Model tersimpan di:", MODEL_PATH.resolve())
print("Metadata tersimpan di:", METADATA_PATH.resolve())

## 11. Tes Load Model dari File `.pkl`

Cell ini memastikan file `.pkl` benar-benar bisa dipakai setelah disimpan.

In [ ]:
loaded_model = joblib.load(MODEL_PATH)
loaded_prediction = loaded_model.predict([[4.33, 2.67, 3.00]])[0]

print("Prediksi dari model reload:", loaded_prediction, "-", LABEL_NAMES[loaded_prediction])
print("Notebook selesai. File model sudah kompatibel dengan web.")